# Anonymize an audio file

Give the model two things:

- a **target** — the audio you want to anonymize. Its *words* are preserved.
- a **reference** — a donor voice. Its *voice* is what you hear in the output.

The output says the same thing in the donor's voice, so the original speaker's identity is not recoverable from the audio.

**Before running:** you need the XTTS v2 checkpoints (~2 GB, once). Either run `anonymize download-model` in a terminal, or uncomment the download cell below.

It runs as-is on the sample pair in `../data/` that ships with the repo. Swap in your own recordings by changing the two paths below.

Sections 1-5 use one donor you name yourself; **section 6** hands it a *pool* instead and lets it pick the donor least like the target, which is what the final runs did.

See [ANONYMIZATION.md](../ANONYMIZATION.md) for the full documentation.

In [ ]:
# Run once if you do not have the checkpoints yet (~2 GB).
# from anonymizer.download import download_model
# download_model()

## 1. Create the anonymizer

Nothing is loaded yet — this is free. The checkpoints load on the first `anonymize` call and stay loaded, so keep this object around.

In [1]:
from anonymizer import Anonymizer

anon = Anonymizer()      # add mode="refine", language="de", device="cpu", ... as needed
anon.config

/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


AnonymizerConfig(model_dir='/home/romolo/VT1/coqui-tts/XTTS_v2.0_original_model_files/', device='cuda', mode='single', language='en', whisper_model='base', iterations=5, threshold=0.6, max_attempts=10, denoise=False, output_sample_rate=24000, reference=None, voice_pool=None, selection='most_distant', select_top_k=10, voice_pool_dir=None)

## 2. Pick your files

Point these at your own audio. For the donor you can also pass a **directory** of wavs — the conditioning is averaged across them, which gives a more stable voice than a single clip.

In [2]:
target = "../data/TARGET_5s.wav"   # the audio to anonymize
reference = "../data/REF.wav"       # the donor voice to hide behind

from pathlib import Path

for role, path in (("target", target), ("reference", reference)):
    if not Path(path).exists():
        raise FileNotFoundError(
            f"{role} audio not found: {path} — run this notebook from examples/, "
            "or point these two variables at your own wav files."
        )

import IPython.display as ipd
print("target (original speaker):")
ipd.display(ipd.Audio(target))
print("reference (donor voice):")
ipd.display(ipd.Audio(reference))

target (original speaker):


reference (donor voice):


## 3. Anonymize

The first call loads the model, so it takes a while; later calls are fast. The transcript is produced with Whisper — pass `text="..."` if you already have one, and `language="de"` (etc.) for non-English audio.

In [3]:
result = anon.anonymize(target, reference=reference)

print("transcript:", result.text)
anon.play(result)

 > Using model: xtts
>> DVAE weights restored from: /home/romolo/VT1/coqui-tts/XTTS_v2.0_original_model_files/dvae.pth
transcript: No sir, I did not mean it so, and I am very, very sorry.


## 4. Save it

In [4]:
anon.save(result, "anonymized.wav")

'anonymized.wav'

## 5. Did it work?

Four numbers. The words should survive (**low** WER, **high** BLEU) while the identity should not (**low** similarity to the original speaker, **high** to the donor).

The absolute similarity values are small even when this works well — what matters is that `reference_similarity` is greater than `target_similarity`.

In [5]:
report = anon.score(result, target)

for name, value in report.to_dict().items():
    print(f"{name:>22}: {value:.3f}")

                   wer: 0.000
                  bleu: 1.000
     target_similarity: 0.198
  reference_similarity: 0.263
       overall_quality: 0.498


## 6. Let it pick the donor for you

Choosing the donor by hand is a gamble: if the donor happens to sound like the person you
are anonymizing, the output barely moves away from them. Hand the anonymizer a **pool** of
voices instead and it picks the **least equal** one — the speaker furthest from *this*
target — for every file you give it. This is what the final runs did.

The rule, in four steps:

1. embed the target and every pool clip with ECAPA2,
2. score each pool **speaker** by the mean cosine similarity of their clips to the target,
3. take the **lowest** score — the voice least like the target's,
4. condition on that speaker's `select_top_k` least-similar clips.

`../data/pool/` holds five speakers from the Common Voice pool those runs drew donors
from (CC-0), three clips each, kept deliberately spread out relative to this target — see
[data/pool/README.md](../data/pool/README.md). One folder per speaker is the whole format:

```
data/pool/
  DE_414560/  common_voice_de_41456000.wav  ...
  DE_414863/  ...
  FR_413330/  ...
  FR_414927/  ...
  IT_416773/  ...
  metadata.csv        <- the same pool as a manifest, with gender and language
```

### Preview the choice

`select_reference` runs **only** ECAPA2 — no XTTS, no Whisper — so you can audit a pool
before committing to a synthesis run. Setting the pool on the anonymizer you already have
keeps the checkpoints loaded.

In [6]:
from pathlib import Path

anon.config.voice_pool = "../data/pool"   # a CSV manifest works here too, see below
anon.config.select_top_k = 2              # the final runs used 10; these speakers have 3 clips each

choice = anon.select_reference(target)

print(f"donor chosen: {choice.speaker_id}  "
      f"(mean similarity {choice.speaker_similarity:+.4f}, out of {choice.candidates} clips)")
for path, similarity in zip(choice.clips, choice.clip_similarities):
    print(f"   {similarity:+.4f}  {Path(path).name}")

print("\nspeakers, furthest first:")
for rank, ranked in enumerate(choice.ranked_speakers, start=1):
    print(f"   {rank}. {ranked.key:<10} {ranked.similarity:+.4f}")

donor chosen: FR_413330  (mean similarity -0.1774, out of 12 clips)
   -0.1984  common_voice_fr_41333036.wav
   -0.1790  common_voice_fr_41333037.wav

speakers, furthest first:
   1. FR_413330  -0.1774
   2. IT_416773  -0.1282
   3. DE_414863  -0.0313
   4. FR_414927  +0.3252


Lower is further away, and small or negative values are normal — what matters is the
order. The spread across these five speakers is real: the two ends of that ranking are two
different people, and only one of them is a safe place to hide.

In [7]:
print(f"the least equal voice — {choice.speaker_id}, what the anonymizer picked:")
ipd.display(ipd.Audio(str(choice.clips[0])))

closest = choice.ranked_speakers[-1]
print(f"the closest voice — {closest.key} at {closest.similarity:+.4f}; "
      f"pick this one by hand and you hide much less:")
ipd.display(ipd.Audio(str(sorted(Path("../data/pool", closest.key).glob("*.wav"))[0])))

the least equal voice — FR_413330, what the anonymizer picked:


the closest voice — FR_414927 at +0.3252; pick this one by hand and you hide much less:


### Filters, and a pool that carries metadata

A directory pool knows only *who* is speaking. Describe the same clips in a **CSV
manifest** and you can filter *before* ranking — by `gender`, by recording `language`, or
both. Only the audio-path column is required; `speaker_id`, `gender` and `language` are
picked up under their usual aliases (`speaker`, `predicted_gender`, `lang`, …), and
relative paths resolve against the CSV's own directory, so a manifest travels with its
audio.

`../data/pool/metadata.csv` is exactly that. Gender is filled in only where the Common
Voice speaker self-reported one — two rows are deliberately blank, because a pool with
missing metadata is the normal case and it is worth seeing what the filter does with it.

Point `voice_pool` at the manifest instead of the folder and the same filters are just
arguments: `anon.select_reference(target, gender="male")`, or `--gender male` on the CLI.

In [8]:
from anonymizer.selection import VoiceSelectionError
from anonymizer.voices import VoicePool

manifest = Path("../data/pool/metadata.csv")
print("\n".join(manifest.read_text().splitlines()[:3] + ["..."]))

# Rank against the manifest with the session you already have, so nothing is re-embedded.
pool = VoicePool.load(str(manifest))
selector = anon.session.selector
target_embedding = anon.session.embed_speaker(target)

for wanted in ("male", "female"):
    picked = selector.select(pool, target_embedding, top_k=2, gender=wanted)
    print(f"\nbest {wanted:<6} donor: {picked.speaker_id} ({picked.speaker_similarity:+.4f})")

italian = selector.select(pool, target_embedding, top_k=2, language="it")
print(f"\nbest Italian donor: {italian.speaker_id} ({italian.speaker_similarity:+.4f})")

try:
    selector.select(pool, target_embedding, top_k=2, language="en")
except VoiceSelectionError as err:
    print("\nand a filter that matches nobody fails loudly rather than "
          "conditioning on nothing:\n  ", err)

path,speaker_id,language
/home/romolo/VT1/coqui-tts/data/pool/DE_414863/common_voice_de_41486308.wav,DE_414863,de
/home/romolo/VT1/coqui-tts/data/pool/DE_414863/common_voice_de_41486309.wav,DE_414863,de
...

restricted to Italian donors: IT_416773 (-0.1282)

and a filter that matches nobody fails loudly rather than conditioning on nothing:
   no donor voices left in the pool after filtering (language='en'). The pool holds 12 clip(s) from 4 speaker(s); check that it carries the metadata you are filtering on.


Note which speakers those filters *can* reach: the two rows with no `gender` in the
manifest are dropped as soon as you filter on gender, because a clip whose metadata does
not record the field cannot be shown to satisfy it. Silently keeping them is how you end
up conditioning on the wrong voices.

### Anonymize against the pool

Call `anonymize` with **no** `reference` — the donor is chosen per file, and which one was
chosen is recorded on the result (and in the batch manifest, so a run stays auditable
afterwards).

In [9]:
pooled = anon.anonymize(target)          # no reference= -> the pool decides

print("donor:", pooled.info["selected_speaker"],
      f"({pooled.info['selected_clips']} clip(s), similarity {pooled.info['selected_speaker_similarity']:+.4f})")
print("transcript:", pooled.text)
anon.play(pooled)

donor: FR_413330 (2 clip(s), similarity -0.1774)
transcript: No sir, I did not mean it so, and I am very, very sorry.


In [10]:
for name, value in anon.score(pooled, target).to_dict().items():
    print(f"{name:>22}: {value:.3f}")

                   wer: 0.000
                  bleu: 1.000
     target_similarity: 0.104
  reference_similarity: 0.208
       overall_quality: 0.494


Compare that `target_similarity` with the one in section 5: same words, but a donor
chosen *against this speaker* rather than picked by hand. In the run recorded here it fell
from about 0.20 to about 0.10 — XTTS samples, so your numbers will differ; the direction
is the point.

Same thing from a terminal:

```bash
anonymize select interview.wav --voice-pool ../data/pool          # preview the choice
anonymize run interview.wav --voice-pool ../data/pool -o out.wav  # and use it
anonymize run interview.wav --voice-pool pool.csv --gender female -o out.wav
```

The pool is embedded once per `Anonymizer` and cached, so anonymizing a folder against a
large pool pays that cost a single time. Selection wants a GPU — ECAPA2 is impractically
slow on CPU.

## Going further

**Higher quality** — slower, wants a GPU:

```python
result = anon.anonymize(target, reference=reference, mode="iterate")  # or "refine"
```

**A whole folder**, resumable, with a manifest CSV:

```python
from anonymizer.batch import anonymize_directory

summary = anonymize_directory(
    "recordings/", "anonymized/", reference=reference, anonymizer=anon, resume=True
)
print(summary["completed"], "files ->", summary["manifest"])
```

**From a terminal:**

```bash
anonymize run interview.wav --reference donor.wav -o out.wav --score
anonymize batch recordings/ --reference donor.wav -o anonymized/ --resume
```

> **Note:** this hides *who* is speaking, not *what* was said. The transcript is
> reproduced verbatim, so identifying words — names, addresses — pass through. See the
> Limitations section in [ANONYMIZATION.md](../ANONYMIZATION.md).